In [31]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader, TensorDataset

In [32]:
np.random.seed(42)
samples = 1_000
data = {
	'a': np.random.randn(samples),
	'b': np.random.randn(samples),
	'c': np.random.randn(samples),
	'd': np.random.choice(['x', 'y', 'z', 'N/A'], samples),
}
df = pd.DataFrame(data)

def gen_click(row):
	return (row['a'] * row['b'] * row['c'] * ((1.0 if row['d'] == 'x' else (-1.0 if row['d'] == 'y' else 0.5)) - 0.1) > 0.0)

df['click'] = df.apply(gen_click, axis=1)

# Features and target
X = df.drop(columns=['click'])
y = df['click']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

In [33]:
df.head()

,a,b,c,d,click
0,0.496714,1.399355,-0.675178,y,True
1,-0.138264,0.924634,-0.144519,N/A,True
2,0.647689,0.059630,-0.792420,y,True
3,1.523030,-0.646937,-0.307962,y,False
4,-0.234153,0.698223,-1.893615,x,True


In [34]:
from collections import defaultdict

numericCols = [
    'a',
    'b',
    'c',
]
categoricalCols = ['d']

def preprocess_fit(df, numericCols=['a', 'b', 'c'], categoricalCols=['d']):
    df = df.copy()
    dic = {
        "numerical_means": {},
        "numerical_stds": {},
        "cat_categories": {},
    }
    # normalize numerical columns
    for col in numericCols:
        dic['numerical_means'][col] = df[col].mean()
        dic['numerical_stds'][col] = df[col].std()
        df[col] = (df[col] - dic['numerical_means'][col]) / dic['numerical_stds'][col]
        df[col] = df[col].fillna(dic['numerical_means'][col])
    for col in categoricalCols:
        # one-hot encode categorical columns
        df[col] = df[col].astype('category')
        cat_categories = df[col].cat.categories
        df = pd.get_dummies(df, columns=[col])
        dic['cat_categories'][col] = cat_categories
    return df, dic
        
    
def preprocess_transform(df, dic):
    df = df.copy()
    for col in numericCols:
        df[col] = (df[col] - dic['numerical_means'][col]) / dic['numerical_stds'][col]
        df[col] = df[col].fillna(dic['numerical_means'][col])
    for col in categoricalCols:
        df[col] = pd.Categorical(df[col], categories=dic['cat_categories'][col])
        df = pd.get_dummies(df, columns=[col])
    return df

# Preprocess datasets
X_train_processed, dic = preprocess_fit(X_train)
X_test_processed = preprocess_transform(X_test, dic)

# Convert to tensors
X_train_tensor = torch.tensor(X_train_processed.values.astype(np.float32))
y_train_tensor = torch.tensor(y_train.values.reshape(-1, 1).astype(np.float32))
X_test_tensor = torch.tensor(X_test_processed.values.astype(np.float32))
y_test_tensor = torch.tensor(y_test.values.reshape(-1, 1).astype(np.float32))

# DataLoader
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32, shuffle=True)

In [35]:
from torchvision import models

# Define a simple DNN model
class DNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=10):
        super(DNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=10):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return self.sigmoid(x)

class ResidualBlock(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(input_dim, output_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(output_dim, output_dim)
        
    def forward(self, x):
        residual = x
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out += residual
        out = self.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=10):
        super(ResNet, self).__init__()
        self.layer0 = nn.Linear(input_dim, hidden_dim)
        self.layer1 = ResidualBlock(hidden_dim, hidden_dim)
        self.layer2 = ResidualBlock(hidden_dim, hidden_dim)
        self.layer3 = ResidualBlock(hidden_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        out = self.layer0(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.fc(out)
        out = self.sigmoid(out)
        return out

# Model initialization
model = ResNet(X_train_tensor.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 1
for epoch in range(epochs):
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

# Evaluation
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor).numpy()
    y_pred_label = (y_pred > 0.5).astype(int)

In [36]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, fbeta_score

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred_label)
precision = precision_score(y_test, y_pred_label)
recall = recall_score(y_test, y_pred_label)
f1 = f1_score(y_test, y_pred_label)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

f0_5 = fbeta_score(y_test, y_pred_label, beta=0.5)
f2 = fbeta_score(y_test, y_pred_label, beta=2.0)

print(f'F0.5 Score: {f0_5:.4f}')
print(f'F2 Score: {f2:.4f}')


Accuracy: 0.4800
Precision: 0.4737
Recall: 0.0874
F1 Score: 0.1475
F0.5 Score: 0.2514
F2 Score: 0.1044


In [37]:
classification_report(y_test, y_pred_label, output_dict=True)

{'False': {'precision': 0.48066298342541436,
  'recall': 0.8969072164948454,
  'f1-score': 0.6258992805755396,
  'support': 97.0},
 'True': {'precision': 0.47368421052631576,
  'recall': 0.08737864077669903,
  'f1-score': 0.14754098360655737,
  'support': 103.0},
 'accuracy': 0.48,
 'macro avg': {'precision': 0.47717359697586503,
  'recall': 0.4921429286357722,
  'f1-score': 0.3867201320910485,
  'support': 200.0},
 'weighted avg': {'precision': 0.47706891538237856,
  'recall': 0.48,
  'f1-score': 0.37954475763651374,
  'support': 200.0}}